# 1. Perkenalan Dataset

**Mahasiswa:** Muhammad Yusuf (APC005D6Y0216)

**Dataset:** Rice (Cammeo & Osmancik) — UCI ML Repository id 545

**Sumber:** https://archive.ics.uci.edu/dataset/545/rice+cammeo+and+osmancik

**Deskripsi:** Dua varietas beras Turki yang diklasifikasikan berdasarkan 7 fitur morfologi hasil pencitraan (area, perimeter, panjang, lebar, eccentricity, convex_area, extent). Total 3810 sampel biner (Cammeo vs Osmancik). Cocok untuk klasifikasi biner dengan akurasi tinggi.

**Alasan pemilihan:**
- Rekomendasi default PRD MSML (UCI 545, akurasi ~0.93).
- Biner, mudah diinterpretasi.
- Cocok untuk studi kasus MLOps end-to-end.

# 2. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
print('library ready')

# 3. Memuat Dataset

In [ ]:
RAW = Path('../Membangun_model/rice_raw.csv')
df = pd.read_csv(RAW)
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe()

# 4. Exploratory Data Analysis (EDA)

In [ ]:
# 4a. Distribusi kelas
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(x='Class', data=df, hue='Class', palette='viridis', legend=False)
ax.set_title('Distribusi kelas')
plt.show()

In [ ]:
# 4b. Pairplot fitur numerik (ringkas)
sns.pairplot(df, hue='Class', diag_kind='hist', corner=True)
plt.suptitle('Pairplot fitur', y=1.02)
plt.show()

In [ ]:
# 4c. Korelasi fitur numerik
numeric = df.select_dtypes(include=np.number)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(numeric.corr(), annot=True, cmap='coolwarm', ax=ax, fmt='.2f')
ax.set_title('Korelasi Pearson')
plt.show()

In [ ]:
# 4d. Boxplot outlier screening
fig, ax = plt.subplots(figsize=(10,5))
df.drop(columns=['Class']).boxplot(ax=ax)
ax.set_title('Boxplot fitur (deteksi outlier)')
plt.xticks(rotation=30, ha='right')
plt.show()

# 5. Data Preprocessing

In [ ]:
# Tahapan preprocessing mengikuti automate_rice.py (idempotent)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

target = 'Class'
features = [c for c in df.columns if c != target]

df_clean = df.dropna().drop_duplicates().reset_index(drop=True)
le = LabelEncoder(); y = le.fit_transform(df_clean[target])
X = df_clean[features].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)

print('train', X_train_s.shape, 'test', X_test_s.shape)
print('classes:', le.classes_)

# 6. Kesimpulan EDA

- Distribusi kelas relatif seimbang (Cammeo & Osmancik, ~1630 + ~2180).
- Fitur area, perimeter, panjang, lebar berkorelasi kuat (multicollinearity).
- Outlier terlihat pada fitur `Extent`; namun model tree-based dapat menanganinya.
- Data sudah bersih (tanpa missing/duplicate setelah dedupe).
- Lanjut ke tahap modelling di folder `Membangun_model/modelling.py`.